In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
import datetime

In [2]:
existing_spark = SparkSession.getActiveSession()

if existing_spark:
    print("Existing Spark session found. Stopping it...")
    existing_spark.stop()

In [3]:
spark = (
        SparkSession.builder
        .appName("Catalog App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    )

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hudi#hudi-spark3.5-bundle_2.12 added as a dependency
software.amazon.awssdk#glue added as a dependency
software.amazon.awssdk#sts added as a dependency
software.amazon.awssdk#s3 added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
software.amazon.awssdk#dynamodb added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-21dd58b1-54fe-45de-bcbf-97b7f454853e;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.a

In [4]:
spark

26/04/28 11:07:25 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [14]:
print("Catalog:", spark.conf.get("spark.sql.catalogImplementation"))

Catalog: hive


##### Set Warehouse dir location either S3 or local path

In [15]:
print("Warehouse Dir:", spark.conf.get("spark.sql.warehouse.dir"))

Warehouse Dir: s3a://shivchoudhury-datasets/warehouse


In [16]:
print("Current Database:", spark.catalog.currentDatabase())

Current Database: my_glue_db


In [17]:
spark.sql("CREATE DATABASE IF NOT EXISTS spark_catalog.catalog_db")

26/04/28 11:12:12 WARN ObjectStore: Failed to get database catalog_db, returning NoSuchObjectException
26/04/28 11:12:12 WARN ObjectStore: Failed to get database catalog_db, returning NoSuchObjectException
26/04/28 11:12:13 WARN ObjectStore: Failed to get database catalog_db, returning NoSuchObjectException


DataFrame[]

In [18]:
spark.sql("USE spark_catalog.catalog_db")

DataFrame[]

In [19]:
print("Current Database:", spark.catalog.currentDatabase())

Current Database: catalog_db


In [20]:
spark.catalog.listTables("catalog_db")

[]

#### Customers Table

In [21]:
# -----------------------
# 2. Customers Data
# -----------------------
customers_data = [
    (1, "Alice Johnson", "alice@example.com", "New York", "2021-01-10"),
    (2, "Bob Smith", "bob@example.com", "Chicago", "2021-02-15"),
    (3, "Charlie Brown", "charlie@example.com", "Los Angeles", "2021-03-20"),
    (4, "David Wilson", "david@example.com", "San Francisco", "2021-04-25"),
    (5, "Eva Davis", "eva@example.com", "Houston", "2021-05-30"),
    (6, "Frank Miller", "frank@example.com", "Miami", "2021-06-10"),
    (7, "Grace Lee", "grace@example.com", "Boston", "2021-07-15"),
    (8, "Hannah Scott", "hannah@example.com", "Seattle", "2021-08-20"),
    (9, "Ian Wright", "ian@example.com", "Denver", "2021-09-25"),
    (10, "Julia Turner", "julia@example.com", "Phoenix", "2021-10-30"),
    (11, "Kevin Harris", "kevin@example.com", "Dallas", "2021-11-05"),
    (12, "Laura King", "laura@example.com", "Austin", "2021-12-10"),
    (13, "Mike Adams", "mike@example.com", "Atlanta", "2022-01-15"),
    (14, "Nina Perez", "nina@example.com", "Orlando", "2022-02-20"),
    (15, "Oscar Clark", "oscar@example.com", "Detroit", "2022-03-25"),
    (16, "Paula Lewis", "paula@example.com", "Columbus", "2022-04-30"),
    (17, "Quinn Hall", "quinn@example.com", "Las Vegas", "2022-05-15"),
    (18, "Rachel Allen", "rachel@example.com", "Philadelphia", "2022-06-20"),
    (19, "Steve Young", "steve@example.com", "San Diego", "2022-07-25"),
    (20, "Tina Evans", "tina@example.com", "Portland", "2022-08-30")
]

customers_df = spark.createDataFrame(customers_data, 
    ["customer_id", "name", "email", "city", "join_date"])

customers_df.write.mode("overwrite").saveAsTable("customers")

26/04/28 11:12:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/28 11:12:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/28 11:12:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/28 11:12:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/28 11:12:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/28 11:12:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/28 11:12:57 WARN MemoryManager: Total allocation exceeds 95.

In [22]:
spark.catalog.listTables("catalog_db")

[Table(name='customers', catalog='spark_catalog', namespace=['catalog_db'], description=None, tableType='MANAGED', isTemporary=False)]

#### Products Table

In [23]:
# -----------------------
# 3. Products Data
# -----------------------
products_data = [
    (1, "Laptop", "Electronics", 850.00, 50),
    (2, "Smartphone", "Electronics", 600.00, 120),
    (3, "Tablet", "Electronics", 300.00, 80),
    (4, "Headphones", "Accessories", 50.00, 200),
    (5, "Keyboard", "Accessories", 30.00, 150),
    (6, "Mouse", "Accessories", 20.00, 170),
    (7, "Monitor", "Electronics", 200.00, 60),
    (8, "Charger", "Accessories", 25.00, 100),
    (9, "Backpack", "Bags", 40.00, 90),
    (10, "Shoes", "Fashion", 70.00, 70),
    (11, "Jacket", "Fashion", 120.00, 40),
    (12, "Watch", "Fashion", 200.00, 30),
    (13, "Desk", "Furniture", 150.00, 25),
    (14, "Chair", "Furniture", 90.00, 35),
    (15, "Lamp", "Furniture", 45.00, 45),
    (16, "Book", "Stationery", 15.00, 300),
    (17, "Pen", "Stationery", 2.00, 500),
    (18, "Notebook", "Stationery", 5.00, 400),
    (19, "Bottle", "Kitchen", 12.00, 250),
    (20, "Mug", "Kitchen", 8.00, 150)
]

products_df = spark.createDataFrame(products_data, 
    ["product_id", "product_name", "category", "price", "stock"])

products_df.write.mode("overwrite").saveAsTable("products")

26/04/28 11:13:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/28 11:13:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/28 11:13:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/28 11:13:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/28 11:13:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/28 11:13:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/28 11:13:50 WARN MemoryManager: Total allocation exceeds 95.

In [24]:
spark.catalog.listTables("catalog_db")

[Table(name='customers', catalog='spark_catalog', namespace=['catalog_db'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='products', catalog='spark_catalog', namespace=['catalog_db'], description=None, tableType='MANAGED', isTemporary=False)]

#### Orders Table

In [25]:
# -----------------------
# 4. Orders Data
# -----------------------
orders_data = [
    (1, 1, 2, 1, "2022-01-05"),
    (2, 2, 5, 2, "2022-01-10"),
    (3, 3, 1, 1, "2022-01-15"),
    (4, 4, 4, 3, "2022-01-20"),
    (5, 5, 3, 1, "2022-01-25"),
    (6, 6, 6, 2, "2022-02-05"),
    (7, 7, 7, 1, "2022-02-10"),
    (8, 8, 8, 1, "2022-02-15"),
    (9, 9, 9, 2, "2022-02-20"),
    (10, 10, 10, 1, "2022-02-25"),
    (11, 11, 12, 1, "2022-03-05"),
    (12, 12, 14, 2, "2022-03-10"),
    (13, 13, 13, 1, "2022-03-15"),
    (14, 14, 11, 1, "2022-03-20"),
    (15, 15, 15, 2, "2022-03-25"),
    (16, 16, 16, 5, "2022-04-05"),
    (17, 17, 18, 3, "2022-04-10"),
    (18, 18, 19, 2, "2022-04-15"),
    (19, 19, 20, 4, "2022-04-20"),
    (20, 20, 17, 10, "2022-04-25")
]

orders_df = spark.createDataFrame(orders_data, 
    ["order_id", "customer_id", "product_id", "quantity", "order_date"])

orders_df.write.mode("overwrite").saveAsTable("orders")

26/04/28 11:14:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/28 11:14:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/28 11:14:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/28 11:14:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/28 11:14:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/28 11:14:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/28 11:14:15 WARN MemoryManager: Total allocation exceeds 95.0

In [26]:
spark.catalog.listTables("catalog_db")

[Table(name='customers', catalog='spark_catalog', namespace=['catalog_db'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='orders', catalog='spark_catalog', namespace=['catalog_db'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='products', catalog='spark_catalog', namespace=['catalog_db'], description=None, tableType='MANAGED', isTemporary=False)]

#### People Table

In [27]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS catalog_db.people (
        id INT,
        name STRING,
        age INT
    )
    ROW FORMAT DELIMITED
    FIELDS TERMINATED BY ','
""")

26/04/28 11:14:45 WARN HiveMetaStore: Location: s3a://shivchoudhury-datasets/warehouse/catalog_db.db/people specified for non-external table:people


DataFrame[]

In [29]:
spark.sql("""
    INSERT INTO catalog_db.people VALUES
    (1, 'Alice', 30),
    (2, 'Bob', 28),
    (3, 'Charlie', 35),
    (4, 'David', 40),
    (5, 'Eva', 25)
""")

26/04/28 11:15:03 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/28 11:15:04 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/28 11:15:04 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/28 11:15:04 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/28 11:15:04 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/28 11:15:04 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
                                                                                

DataFrame[]

In [30]:
spark.catalog.listTables("catalog_db")

[Table(name='customers', catalog='spark_catalog', namespace=['catalog_db'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='orders', catalog='spark_catalog', namespace=['catalog_db'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='people', catalog='spark_catalog', namespace=['catalog_db'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='products', catalog='spark_catalog', namespace=['catalog_db'], description=None, tableType='MANAGED', isTemporary=False)]

In [31]:
spark.sql("select * from catalog_db.people").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 28|
|  3|Charlie| 35|
|  4|  David| 40|
|  5|    Eva| 25|
+---+-------+---+



In [32]:
spark.stop()